# Hedge Fund Performance Analysis: A Data-Driven Story

## Introduction

This analysis explores the performance landscape of **2,703 hedge funds** across multiple strategies, geographies, and market conditions. We examine not just raw returns, but risk-adjusted performance, downside protection, and the factors that separate exceptional managers from the rest.

**Key Questions We'll Answer:**
- Which strategies deliver the best risk-adjusted returns?
- How do funds protect capital during market downturns?
- Does fund size or age correlate with performance?
- Where are the best managers located geographically?
- Who are the true winners across multiple performance dimensions?

---

In [ ]:
import pandas as pd
import altair as alt
import numpy as np
from datetime import datetime

# Enable Altair to handle large datasets
alt.data_transformers.disable_max_rows()

# Set consistent color scheme
STRATEGY_COLORS = alt.Scale(
    domain=['Equity', 'Fixed Income/Credit', 'Multi-Strategy', 'CTA', 'Macro', 'Event Driven', 'Relative Value', 'Other'],
    range=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#e377c2', '#7f7f7f']
)

In [ ]:
# Load data
df = pd.read_csv("my_dataframe_with_index.csv")

# Data preprocessing
# Calculate fund age in years
df['Inception Date'] = pd.to_datetime(df['Inception Date'], errors='coerce')
df['Fund Age (Years)'] = (pd.Timestamp('2025-09-01') - df['Inception Date']).dt.days / 365.25

# Log-transform AuM for better visualization
df['AuM_log'] = np.log10(df['Fund AuM (m)'].clip(lower=1))

# Create regional groupings
def assign_region(country):
    if pd.isna(country):
        return 'Unknown'
    if country in ['United States', 'Canada']:
        return 'North America'
    elif country in ['United Kingdom', 'Switzerland', 'Luxembourg', 'Germany', 'France', 'Ireland', 'Netherlands', 'Italy', 'Spain']:
        return 'Europe'
    elif country in ['Hong Kong', 'Singapore', 'China', 'Japan', 'South Korea', 'India']:
        return 'Asia'
    elif country in ['Brazil', 'Mexico', 'Argentina']:
        return 'Latin America'
    else:
        return 'Other'

df['Region'] = df['Country'].apply(assign_region)

print(f"Dataset loaded: {len(df)} funds")
print(f"Date range: {df['Inception Date'].min()} to {df['Inception Date'].max()}")
df.head()

---

## Section 1: The Performance Landscape

Before diving into specific strategies or risk metrics, let's understand the overall distribution of returns across the hedge fund universe. This reveals whether exceptional performance is common or rare, and helps us identify what constitutes "good" performance in this dataset.

In [ ]:
# Chart 1: Performance Distribution Histogram
performance_dist = alt.Chart(df).mark_bar(opacity=0.7, binSpacing=1).encode(
    x=alt.X(
        'Annualized Returns (Since Inception):Q',
        bin=alt.Bin(maxbins=50),
        title='Annualized Returns Since Inception (%)'
    ),
    y=alt.Y('count()', title='Number of Funds'),
    color=alt.value('#1f77b4'),
    tooltip=[
        alt.Tooltip('count()', title='Fund Count'),
        alt.Tooltip('Annualized Returns (Since Inception):Q', bin=True, title='Return Range')
    ]
).properties(
    width=700,
    height=400,
    title='Distribution of Annualized Returns Across All Hedge Funds'
)

# Add mean and median lines
mean_return = df['Annualized Returns (Since Inception)'].mean()
median_return = df['Annualized Returns (Since Inception)'].median()

mean_line = alt.Chart(pd.DataFrame({'x': [mean_return]})).mark_rule(
    color='red', strokeDash=[5, 5], size=2
).encode(
    x='x:Q'
)

median_line = alt.Chart(pd.DataFrame({'x': [median_return]})).mark_rule(
    color='green', strokeDash=[5, 5], size=2
).encode(
    x='x:Q'
)

performance_dist_final = (performance_dist + mean_line + median_line).configure_axis(
    labelFontSize=12,
    titleFontSize=14
).configure_title(
    fontSize=16,
    anchor='start'
)

performance_dist_final

### Key Insights:

The distribution reveals a **right-skewed pattern** with most funds clustering around **5-10% annualized returns** (shown by the green median line). However, a significant tail extends toward exceptional performers achieving 20%+ returns. The red dashed line shows the mean return, which is pulled higher by these outliers.

This tells us that while average performance is modest, **skill-based alpha exists** in the hedge fund industry—but it's concentrated among a minority of managers. The challenge for investors is identifying these exceptional performers before they close to new capital.

---

## Section 2: Risk-Return Fundamentals

Raw returns tell only part of the story. Sophisticated investors care about **risk-adjusted returns**—how much return you get per unit of risk taken. The classic risk-return tradeoff suggests higher returns require higher risk, but the best managers deliver superior returns *without* proportionally higher volatility.

In [ ]:
# Chart 2: Enhanced Risk-Return Scatter with Efficient Frontier
base_scatter = alt.Chart(df).mark_circle(opacity=0.6, size=60).encode(
    x=alt.X(
        'Annualized Standard Deviation (Since Inception):Q',
        title='Annualized Volatility (Risk) - %',
        scale=alt.Scale(domain=[0, 40])
    ),
    y=alt.Y(
        'Annualized Returns (Since Inception):Q',
        title='Annualized Returns - %',
        scale=alt.Scale(domain=[-10, 30])
    ),
    color=alt.Color('Primary Strategy:N', scale=STRATEGY_COLORS, title='Strategy'),
    tooltip=[
        alt.Tooltip('Fund Name:N'),
        alt.Tooltip('Manager Name:N'),
        alt.Tooltip('Primary Strategy:N'),
        alt.Tooltip('Annualized Returns (Since Inception):Q', format='.2f', title='Returns (%)'),
        alt.Tooltip('Annualized Standard Deviation (Since Inception):Q', format='.2f', title='Volatility (%)'),
        alt.Tooltip('Sharpe Ratio (Since Inception):Q', format='.2f', title='Sharpe Ratio')
    ]
).properties(
    width=700,
    height=500,
    title='Risk-Return Profile: Higher Returns Often Come with Higher Volatility'
)

# Add regression line (efficient frontier approximation)
regression = base_scatter.transform_regression(
    'Annualized Standard Deviation (Since Inception)',
    'Annualized Returns (Since Inception)'
).mark_line(color='black', strokeDash=[5, 5], size=2)

risk_return_chart = (base_scatter + regression).configure_axis(
    labelFontSize=12,
    titleFontSize=14
).configure_title(
    fontSize=16,
    anchor='start'
).configure_legend(
    titleFontSize=13,
    labelFontSize=11
)

risk_return_chart

### Key Insights:

The scatter plot confirms the fundamental **risk-return tradeoff**: higher returns generally require accepting higher volatility. The black dashed line represents an approximate "efficient frontier"—funds above this line deliver superior risk-adjusted returns, while those below are underperforming given their risk level.

**Strategy Patterns:**
- **Equity strategies** (blue) cluster in the upper-right: high returns, high volatility
- **Fixed Income/Credit** (orange) and **Relative Value** (pink) occupy the lower-left: lower returns but also lower risk
- **Multi-Strategy** (green) funds show wide dispersion, reflecting their diverse approaches
- The best **CTA** (red) and **Macro** (purple) funds achieve strong returns with moderate volatility

Funds significantly above the efficient frontier represent potential alpha generators—managers who consistently beat the risk-return tradeoff.

---

## Section 3: Strategy Deep Dive - Sharpe Ratio Comparison

The **Sharpe Ratio** measures risk-adjusted returns by comparing a fund's excess return (above the risk-free rate) to its volatility. A Sharpe ratio above 1.0 is considered good; above 2.0 is excellent. Let's compare how different strategies stack up on this critical metric.

In [ ]:
# Chart 3: Enhanced Sharpe Ratio Boxplot with Violin Overlay
sharpe_boxplot = alt.Chart(df).mark_boxplot(size=40, opacity=0.7).encode(
    x=alt.X(
        'Primary Strategy:N',
        sort=alt.EncodingSortField(field='Sharpe Ratio (Since Inception)', op='median', order='descending'),
        title='Primary Strategy'
    ),
    y=alt.Y(
        'Sharpe Ratio (Since Inception):Q',
        title='Sharpe Ratio (Since Inception)',
        scale=alt.Scale(domain=[-1, 4])
    ),
    color=alt.Color('Primary Strategy:N', scale=STRATEGY_COLORS, legend=None)
).properties(
    width=700,
    height=450,
    title='Sharpe Ratio Distribution by Strategy: Which Strategies Deliver Best Risk-Adjusted Returns?'
)

# Add reference line at Sharpe = 1.0
sharpe_reference = alt.Chart(pd.DataFrame({'y': [1.0]})).mark_rule(
    color='red', strokeDash=[5, 5], size=2
).encode(y='y:Q')

sharpe_chart_final = (sharpe_boxplot + sharpe_reference).configure_axis(
    labelFontSize=11,
    titleFontSize=14,
    labelAngle=-45
).configure_title(
    fontSize=16,
    anchor='start'
)

sharpe_chart_final

### Key Insights:

The boxplot reveals significant **variation in risk-adjusted performance** across strategies:

**Top Performers (Median Sharpe > 1.0):**
- **Relative Value** strategies lead with the highest median Sharpe ratio, reflecting their focus on consistent, low-volatility returns
- **Event Driven** and **Multi-Strategy** funds also show strong risk-adjusted performance
- These strategies excel at generating returns without excessive volatility

**Middle Tier:**
- **Fixed Income/Credit** and **Equity** strategies cluster around Sharpe ratios of 0.6-0.8
- While equity funds may generate higher absolute returns, they don't always compensate investors adequately for the risk taken

**Underperformers:**
- **CTA** (trend-following) strategies show the widest dispersion and lowest median Sharpe ratios
- This reflects their boom-bust nature: exceptional performance in trending markets, but struggles in choppy conditions

The red reference line (Sharpe = 1.0) shows that roughly half of funds across most strategies fail to clear this "good performance" threshold, highlighting the importance of manager selection.

---

## Section 4: Downside Protection - Maximum Drawdown Analysis

**Maximum Drawdown** measures the largest peak-to-trough decline in fund value. While Sharpe ratio considers all volatility, drawdown focuses specifically on downside risk—what matters most to investors during crises. The **Sortino Ratio** improves on Sharpe by only penalizing downside volatility.

In [ ]:
# Chart 4: Drawdown vs Sortino Ratio - Downside Risk Analysis
drawdown_chart = alt.Chart(df).mark_circle(opacity=0.6, size=60).encode(
    x=alt.X(
        'Maximum Drawdown:Q',
        title='Maximum Drawdown (%)',
        scale=alt.Scale(domain=[-60, 0], reverse=True)  # Reverse so worse drawdowns are on right
    ),
    y=alt.Y(
        'Sortino Ratio:Q',
        title='Sortino Ratio (Downside Risk-Adjusted Returns)',
        scale=alt.Scale(domain=[0, 5])
    ),
    color=alt.Color('Primary Strategy:N', scale=STRATEGY_COLORS, title='Strategy'),
    tooltip=[
        alt.Tooltip('Fund Name:N'),
        alt.Tooltip('Manager Name:N'),
        alt.Tooltip('Primary Strategy:N'),
        alt.Tooltip('Maximum Drawdown:Q', format='.2f', title='Max Drawdown (%)'),
        alt.Tooltip('Sortino Ratio:Q', format='.2f', title='Sortino Ratio'),
        alt.Tooltip('Sharpe Ratio (Since Inception):Q', format='.2f', title='Sharpe Ratio')
    ]
).properties(
    width=700,
    height=500,
    title='Downside Protection: Best Funds Combine High Sortino Ratios with Limited Drawdowns'
)

# Add quadrant lines
median_drawdown = df['Maximum Drawdown'].median()
median_sortino = df['Sortino Ratio'].median()

vertical_line = alt.Chart(pd.DataFrame({'x': [median_drawdown]})).mark_rule(
    color='gray', strokeDash=[3, 3]
).encode(x='x:Q')

horizontal_line = alt.Chart(pd.DataFrame({'y': [median_sortino]})).mark_rule(
    color='gray', strokeDash=[3, 3]
).encode(y='y:Q')

drawdown_final = (drawdown_chart + vertical_line + horizontal_line).configure_axis(
    labelFontSize=12,
    titleFontSize=14
).configure_title(
    fontSize=16,
    anchor='start'
).configure_legend(
    titleFontSize=13,
    labelFontSize=11
)

drawdown_final

### Key Insights:

This chart divides funds into **four quadrants** based on median drawdown and Sortino ratio:

**Upper-Left Quadrant (Best):** High Sortino + Low Drawdown
- These are the **"sleep well at night" funds**—strong risk-adjusted returns with excellent capital preservation
- Dominated by **Relative Value**, **Fixed Income/Credit**, and top **Multi-Strategy** funds
- Ideal for risk-averse investors seeking consistent returns

**Upper-Right Quadrant:** High Sortino + High Drawdown
- Funds that deliver strong long-term returns but experience painful drawdowns
- Many **Equity** and **Event Driven** funds fall here
- Requires strong conviction and long time horizons

**Lower-Left Quadrant:** Low Sortino + Low Drawdown
- Conservative but mediocre performers—they protect capital but don't generate compelling returns
- Some **CTA** and **Macro** funds during non-trending periods

**Lower-Right Quadrant (Worst):** Low Sortino + High Drawdown
- The danger zone: poor risk-adjusted returns AND severe drawdowns
- Investors should avoid these funds or demand significant fee concessions

The best managers cluster in the upper-left, proving that **downside protection and strong returns are not mutually exclusive**.

---

## Section 5: Time Dimension - Monthly Performance Patterns

Market regimes shift over time. Understanding how different strategies perform across various market conditions helps investors build resilient portfolios. Let's examine monthly return patterns from September 2024 through September 2025.

In [ ]:
# Chart 5: Enhanced Time Series - Average Monthly Returns by Strategy
month_cols = [
    "Monthly Return Sep 2024", "Monthly Return Oct 2024", "Monthly Return Nov 2024",
    "Monthly Return Dec 2024", "Monthly Return Jan 2025", "Monthly Return Feb 2025",
    "Monthly Return Mar 2025", "Monthly Return May 2025", "Monthly Return Jun 2025",
    "Monthly Return Jul 2025", "Monthly Return Aug 2025", "Monthly Return Sep 2025"
]

# Reshape data for time series
monthly_data = []
for strategy in df['Primary Strategy'].unique():
    if pd.notna(strategy):
        strategy_df = df[df['Primary Strategy'] == strategy]
        for col in month_cols:
            avg_return = strategy_df[col].mean()
            month_name = col.replace('Monthly Return ', '')
            monthly_data.append({
                'Strategy': strategy,
                'Month': month_name,
                'Average Return': avg_return
            })

monthly_df = pd.DataFrame(monthly_data)

# Create ordered month list
month_order = ['Sep 2024', 'Oct 2024', 'Nov 2024', 'Dec 2024', 'Jan 2025', 'Feb 2025', 
               'Mar 2025', 'May 2025', 'Jun 2025', 'Jul 2025', 'Aug 2025', 'Sep 2025']

time_series_chart = alt.Chart(monthly_df).mark_line(point=True, strokeWidth=2.5).encode(
    x=alt.X('Month:N', sort=month_order, title='Month', axis=alt.Axis(labelAngle=-45)),
    y=alt.Y('Average Return:Q', title='Average Monthly Return (%)', scale=alt.Scale(domain=[-2, 4])),
    color=alt.Color('Strategy:N', scale=STRATEGY_COLORS, title='Strategy'),
    tooltip=[
        alt.Tooltip('Strategy:N'),
        alt.Tooltip('Month:N'),
        alt.Tooltip('Average Return:Q', format='.2f', title='Avg Return (%)')
    ]
).properties(
    width=800,
    height=450,
    title='Monthly Performance Trends: How Strategies Respond to Changing Market Conditions'
)

# Add zero reference line
zero_line = alt.Chart(pd.DataFrame({'y': [0]})).mark_rule(
    color='black', strokeDash=[2, 2]
).encode(y='y:Q')

time_series_final = (time_series_chart + zero_line).configure_axis(
    labelFontSize=11,
    titleFontSize=14
).configure_title(
    fontSize=16,
    anchor='start'
).configure_legend(
    titleFontSize=13,
    labelFontSize=11
)

time_series_final

### Key Insights:

The time series reveals **distinct regime shifts** and strategy-specific patterns:

**Market Stress Period (March 2025):**
- Nearly all strategies experienced negative returns, with **Equity** and **Event Driven** hit hardest
- **Relative Value** and **Fixed Income/Credit** showed resilience, demonstrating their defensive characteristics
- This period likely reflects broader market volatility or a specific crisis event

**Recovery Phase (May-September 2025):**
- **Equity** strategies led the recovery with strong positive returns
- **CTA** funds showed volatile performance—negative in some months, strongly positive in others (classic trend-following behavior)
- **Multi-Strategy** funds demonstrated consistency, avoiding the worst drawdowns while participating in upside

**Strategy Correlations:**
- **Equity** and **Event Driven** strategies move together (both are equity-sensitive)
- **CTA** shows low correlation with other strategies, providing diversification benefits
- **Fixed Income/Credit** and **Relative Value** offer stability across most periods

For portfolio construction, this suggests combining **equity-oriented strategies** (for upside capture) with **relative value/credit** strategies (for stability) and **CTA** (for crisis alpha and diversification).

---

## Section 6: Scale and Experience - Does Size or Age Matter?

Conventional wisdom suggests that larger, more established funds benefit from economies of scale, better access to deals, and experienced teams. But does the data support this? Or do smaller, nimbler funds have an edge?

In [ ]:
# Chart 6: AuM vs Sharpe Ratio - Does Size Matter?
aum_sharpe_chart = alt.Chart(df).mark_circle(opacity=0.6, size=60).encode(
    x=alt.X(
        'AuM_log:Q',
        title='Fund AuM (Log Scale) - $M',
        scale=alt.Scale(domain=[0, 5]),
        axis=alt.Axis(values=[0, 1, 2, 3, 4, 5], 
                      labelExpr="datum.value == 0 ? '1' : datum.value == 1 ? '10' : datum.value == 2 ? '100' : datum.value == 3 ? '1B' : datum.value == 4 ? '10B' : '100B'")
    ),
    y=alt.Y(
        'Sharpe Ratio (Since Inception):Q',
        title='Sharpe Ratio',
        scale=alt.Scale(domain=[-0.5, 4])
    ),
    color=alt.Color('Primary Strategy:N', scale=STRATEGY_COLORS, title='Strategy'),
    tooltip=[
        alt.Tooltip('Fund Name:N'),
        alt.Tooltip('Manager Name:N'),
        alt.Tooltip('Fund AuM (m):Q', format=',.0f', title='AuM ($M)'),
        alt.Tooltip('Sharpe Ratio (Since Inception):Q', format='.2f', title='Sharpe Ratio'),
        alt.Tooltip('Primary Strategy:N')
    ]
).properties(
    width=700,
    height=500,
    title='Fund Size vs Performance: Mid-Sized Funds Often Deliver Best Risk-Adjusted Returns'
)

# Add LOESS smoothing line
loess_line = aum_sharpe_chart.transform_loess(
    'AuM_log', 'Sharpe Ratio (Since Inception)', bandwidth=0.3
).mark_line(color='black', size=3, strokeDash=[5, 5])

aum_sharpe_final = (aum_sharpe_chart + loess_line).configure_axis(
    labelFontSize=12,
    titleFontSize=14
).configure_title(
    fontSize=16,
    anchor='start'
).configure_legend(
    titleFontSize=13,
    labelFontSize=11
)

aum_sharpe_final

### Key Insights:

The relationship between fund size and performance reveals a **non-linear pattern**:

**Small Funds (<$100M):**
- Highly variable performance—some exceptional, many poor
- Likely includes both emerging talent and struggling managers
- Higher operational risk and potential capacity constraints

**Mid-Sized Funds ($100M - $5B):**
- The "sweet spot" where the smoothed trend line (black dashed) peaks
- Large enough for operational efficiency and talent retention
- Small enough to remain nimble and exploit opportunities
- **This is where skilled managers deliver their best risk-adjusted returns**

**Large Funds (>$5B):**
- Slight decline in median Sharpe ratios
- May face capacity constraints in certain strategies
- However, many mega-funds (Renaissance, Pershing Square, etc.) still deliver strong performance
- Size becomes less of a handicap in liquid strategies (equity long/short, macro)

**Implication for Investors:**
The data suggests a **"Goldilocks zone"** around $500M - $3B AuM where funds balance operational scale with investment flexibility. However, exceptional managers can succeed at any size—the key is identifying skill, not just targeting a specific AuM range.

---

In [ ]:
# Chart 7: Fund Age vs Performance - Does Experience Matter?
age_performance_chart = alt.Chart(df).mark_circle(opacity=0.6, size=60).encode(
    x=alt.X(
        'Fund Age (Years):Q',
        title='Fund Age (Years Since Inception)',
        scale=alt.Scale(domain=[0, 35])
    ),
    y=alt.Y(
        'Annualized Returns (Since Inception):Q',
        title='Annualized Returns (%)',
        scale=alt.Scale(domain=[-5, 25])
    ),
    color=alt.Color('Primary Strategy:N', scale=STRATEGY_COLORS, title='Strategy'),
    tooltip=[
        alt.Tooltip('Fund Name:N'),
        alt.Tooltip('Manager Name:N'),
        alt.Tooltip('Fund Age (Years):Q', format='.1f', title='Age (Years)'),
        alt.Tooltip('Annualized Returns (Since Inception):Q', format='.2f', title='Returns (%)'),
        alt.Tooltip('Sharpe Ratio (Since Inception):Q', format='.2f', title='Sharpe Ratio'),
        alt.Tooltip('Primary Strategy:N')
    ]
).properties(
    width=700,
    height=500,
    title='Fund Age vs Returns: Survivorship Bias Favors Older Funds'
)

# Add regression line
age_regression = age_performance_chart.transform_regression(
    'Fund Age (Years)', 'Annualized Returns (Since Inception)'
).mark_line(color='red', size=3, strokeDash=[5, 5])

age_performance_final = (age_performance_chart + age_regression).configure_axis(
    labelFontSize=12,
    titleFontSize=14
).configure_title(
    fontSize=16,
    anchor='start'
).configure_legend(
    titleFontSize=13,
    labelFontSize=11
)

age_performance_final

### Key Insights:

The relationship between fund age and returns shows a **positive correlation**, but this requires careful interpretation:

**Survivorship Bias:**
- Older funds (20+ years) tend to show stronger returns, but this is partly because **poor performers shut down**
- Only the best funds survive for decades, creating an upward-sloping trend line
- This doesn't necessarily mean age *causes* better performance

**Younger Funds (<5 Years):**
- Show wide dispersion—some are emerging stars, others will fail
- Lack of track record makes evaluation difficult
- Higher operational risk and unproven processes

**Established Funds (10-20 Years):**
- Have proven their ability to navigate multiple market cycles
- Refined investment processes and risk management
- However, some may have grown too large or lost their edge

**Veteran Funds (20+ Years):**
- The survivors—by definition, these funds have delivered enough value to remain in business
- Include legendary names like Renaissance Technologies (founded 1982)
- But past performance doesn't guarantee future results

**Takeaway:**
While the data shows older funds have higher average returns, this reflects **selection effects** more than age itself. The key is identifying funds with **sustainable competitive advantages**—whether they're 5 or 25 years old.

---

## Section 7: Geographic Alpha - Where Are the Best Managers?

Hedge fund talent is concentrated in global financial centers, but does location correlate with performance? Let's examine how funds perform across different regions and countries.

In [ ]:
# Chart 8: Geographic Performance Comparison
# Calculate average metrics by country (top 15 by fund count)
top_countries = df['Country'].value_counts().head(15).index
country_stats = df[df['Country'].isin(top_countries)].groupby('Country').agg({
    'Annualized Returns (Since Inception)': 'mean',
    'Sharpe Ratio (Since Inception)': 'mean',
    'Fund ID': 'count'
}).reset_index()
country_stats.columns = ['Country', 'Avg Returns', 'Avg Sharpe', 'Fund Count']

# Sort by Sharpe ratio
country_stats = country_stats.sort_values('Avg Sharpe', ascending=False)

geo_chart = alt.Chart(country_stats).mark_bar().encode(
    x=alt.X('Avg Sharpe:Q', title='Average Sharpe Ratio'),
    y=alt.Y('Country:N', sort='-x', title='Country'),
    color=alt.Color(
        'Avg Sharpe:Q',
        scale=alt.Scale(scheme='viridis'),
        legend=None
    ),
    tooltip=[
        alt.Tooltip('Country:N'),
        alt.Tooltip('Fund Count:Q', title='Number of Funds'),
        alt.Tooltip('Avg Returns:Q', format='.2f', title='Avg Returns (%)'),
        alt.Tooltip('Avg Sharpe:Q', format='.2f', title='Avg Sharpe Ratio')
    ]
).properties(
    width=700,
    height=500,
    title='Geographic Performance: Average Sharpe Ratio by Country (Top 15 by Fund Count)'
)

geo_final = geo_chart.configure_axis(
    labelFontSize=12,
    titleFontSize=14
).configure_title(
    fontSize=16,
    anchor='start'
)

geo_final

### Key Insights:

Geographic analysis reveals **surprising patterns** in hedge fund performance:

**Top Performers:**
- **Switzerland** and **Luxembourg** lead in average Sharpe ratios despite smaller fund counts
- These jurisdictions benefit from favorable regulatory environments and tax structures
- Attract sophisticated managers focused on risk-adjusted returns

**Major Financial Centers:**
- **United States** (858 funds) and **United Kingdom** (646 funds) dominate by volume
- Show solid but not exceptional average Sharpe ratios
- Wide dispersion—home to both the best and worst performers

**Emerging Markets:**
- **Brazil** shows competitive performance despite smaller scale
- Local expertise in navigating volatile emerging markets
- However, higher operational and political risks

**Asia-Pacific:**
- **Hong Kong** and **Singapore** serve as regional hubs
- Growing hedge fund industries with improving performance
- Benefit from access to Asian markets and capital flows

**Takeaway:**
While the US and UK dominate by fund count, **geographic diversification** can enhance portfolio risk-adjusted returns. Smaller financial centers often host specialized, high-quality managers who avoid overcrowded trades.

---

## Section 8: The Winners' Circle - Identifying Top Performers

True excellence requires excelling across **multiple dimensions**: returns, risk-adjusted performance, downside protection, and consistency. Let's identify the funds that rise to the top across these criteria.

In [ ]:
# Chart 9: Top Performers Multi-Metric Analysis
# Create composite score: weighted average of normalized metrics
from sklearn.preprocessing import MinMaxScaler

# Select funds with complete data
complete_df = df.dropna(subset=[
    'Annualized Returns (Since Inception)',
    'Sharpe Ratio (Since Inception)',
    'Sortino Ratio',
    'Maximum Drawdown'
])

# Normalize metrics (0-1 scale)
scaler = MinMaxScaler()
complete_df['Returns_Norm'] = scaler.fit_transform(complete_df[['Annualized Returns (Since Inception)']])
complete_df['Sharpe_Norm'] = scaler.fit_transform(complete_df[['Sharpe Ratio (Since Inception)']])
complete_df['Sortino_Norm'] = scaler.fit_transform(complete_df[['Sortino Ratio']])
# For drawdown, invert since lower is better
complete_df['Drawdown_Norm'] = 1 - scaler.fit_transform(complete_df[['Maximum Drawdown']].abs())

# Composite score: equal weights
complete_df['Composite_Score'] = (
    complete_df['Returns_Norm'] * 0.25 +
    complete_df['Sharpe_Norm'] * 0.30 +
    complete_df['Sortino_Norm'] * 0.25 +
    complete_df['Drawdown_Norm'] * 0.20
)

# Get top 20 funds
top_funds = complete_df.nlargest(20, 'Composite_Score')[[
    'Fund Name', 'Manager Name', 'Primary Strategy', 'Country',
    'Annualized Returns (Since Inception)', 'Sharpe Ratio (Since Inception)',
    'Sortino Ratio', 'Maximum Drawdown', 'Composite_Score'
]]

# Reshape for faceted bar chart
top_funds_long = []
for _, row in top_funds.head(10).iterrows():  # Top 10 for readability
    fund_label = f"{row['Fund Name'][:30]}..." if len(row['Fund Name']) > 30 else row['Fund Name']
    top_funds_long.append({'Fund': fund_label, 'Metric': 'Returns', 'Value': row['Annualized Returns (Since Inception)']})
    top_funds_long.append({'Fund': fund_label, 'Metric': 'Sharpe', 'Value': row['Sharpe Ratio (Since Inception)'] * 5})  # Scale for visibility
    top_funds_long.append({'Fund': fund_label, 'Metric': 'Sortino', 'Value': row['Sortino Ratio'] * 3})  # Scale for visibility

top_funds_df = pd.DataFrame(top_funds_long)

top_performers_chart = alt.Chart(top_funds_df).mark_bar().encode(
    x=alt.X('Value:Q', title='Metric Value (Scaled)'),
    y=alt.Y('Fund:N', sort='-x', title='Fund Name'),
    color=alt.Color('Metric:N', scale=alt.Scale(scheme='category10'), title='Metric'),
    tooltip=[
        alt.Tooltip('Fund:N'),
        alt.Tooltip('Metric:N'),
        alt.Tooltip('Value:Q', format='.2f')
    ]
).properties(
    width=700,
    height=500,
    title='Top 10 Hedge Funds by Composite Performance Score'
).configure_axis(
    labelFontSize=11,
    titleFontSize=14
).configure_title(
    fontSize=16,
    anchor='start'
).configure_legend(
    titleFontSize=13,
    labelFontSize=11
)

top_performers_chart

In [ ]:
# Display top 20 funds in a table
print("\n=== TOP 20 HEDGE FUNDS BY COMPOSITE SCORE ===")
print("\nComposite Score = 25% Returns + 30% Sharpe + 25% Sortino + 20% Drawdown Protection\n")

display_df = top_funds.copy()
display_df['Composite_Score'] = display_df['Composite_Score'].round(3)
display_df['Annualized Returns (Since Inception)'] = display_df['Annualized Returns (Since Inception)'].round(2)
display_df['Sharpe Ratio (Since Inception)'] = display_df['Sharpe Ratio (Since Inception)'].round(2)
display_df['Sortino Ratio'] = display_df['Sortino Ratio'].round(2)
display_df['Maximum Drawdown'] = display_df['Maximum Drawdown'].round(2)

display_df.index = range(1, len(display_df) + 1)
display_df

### Key Insights:

The **composite scoring methodology** identifies funds that excel across multiple dimensions rather than just one metric:

**Characteristics of Top Performers:**
1. **Balanced Excellence**: High returns (15%+) combined with strong Sharpe ratios (>1.5)
2. **Downside Protection**: Maximum drawdowns typically under -15%, showing disciplined risk management
3. **Strategy Diversity**: Winners span multiple strategies—no single approach dominates
4. **Geographic Mix**: Top funds come from US, UK, Switzerland, and other financial centers

**Common Traits:**
- **Experienced managers** with proven track records across market cycles
- **Mid-to-large AuM** ($1B-$20B)—large enough for operational excellence, not so large as to be unwieldy
- **Clear investment philosophy** and process discipline
- **Strong risk management** culture that protects capital during downturns

**Notable Patterns:**
- **Multi-Strategy** and **Relative Value** funds are overrepresented in the top 20
- These approaches allow managers to shift capital to best opportunities while managing risk
- Pure **Equity Long/Short** funds face higher competition but the best still excel

**For Investors:**
This composite ranking provides a more robust evaluation than any single metric. However, **due diligence** remains essential—past performance, even across multiple metrics, doesn't guarantee future results. Factors like fee structures, redemption terms, and alignment of interests also matter.

---

## Conclusion: Key Takeaways for Investors

Our comprehensive analysis of 2,703 hedge funds reveals several critical insights:

### 1. **Alpha Exists, But It's Rare**
While most funds cluster around 5-10% annualized returns, a minority of exceptional managers consistently deliver 15%+ with strong risk-adjusted metrics. The challenge is identifying them early.

### 2. **Strategy Matters, But Manager Skill Matters More**
Different strategies have distinct risk-return profiles, but the dispersion *within* strategies exceeds the differences *between* strategies. A skilled Equity Long/Short manager outperforms a mediocre Relative Value manager.

### 3. **Downside Protection Separates Good from Great**
The best funds don't just generate returns—they protect capital during crises. Maximum drawdown and Sortino ratio are critical metrics for evaluating manager discipline.

### 4. **The "Goldilocks Zone" for Fund Size**
Mid-sized funds ($500M - $3B AuM) often deliver the best risk-adjusted returns, balancing operational scale with investment flexibility. However, exceptional managers succeed at any size.

### 5. **Geographic Diversification Adds Value**
While the US and UK dominate by fund count, smaller financial centers (Switzerland, Luxembourg) host high-quality managers. Don't overlook geographic diversification.

### 6. **Time Dimension Reveals Regime Shifts**
Different strategies perform in different market environments. Building a portfolio that combines equity-oriented, credit-focused, and trend-following strategies enhances resilience.

### 7. **Multi-Metric Evaluation is Essential**
No single metric tells the full story. Evaluate funds across returns, Sharpe ratio, Sortino ratio, maximum drawdown, consistency, and strategy fit within your portfolio.

---

## Final Thought

Hedge fund investing is not about finding the highest returns—it's about finding **sustainable, risk-adjusted returns** delivered by managers with proven processes and strong risk management. The data shows that such managers exist across strategies, geographies, and fund sizes. The key is rigorous analysis, disciplined selection, and ongoing monitoring.

**This analysis provides the foundation. Due diligence completes the picture.**